# Phase 9 — expert-usage fingerprint of the tipped mode (MoE router hooks, Colab A100)

White-box instrument: it teacher-forces the **existing** WeirdChat transcripts through
the self-hosted Qwen3.6-35B-A3B MoE, hooks every MoE **gate/router**, and records which
top-k experts fire at every token. From that it measures, per transcript:

* **expert breadth** — how many *distinct* experts a routing site uses across the answer,
* **routing novelty per token** — how many of a token's experts are new vs the preceding
  window (a routing *shift*), and specifically the novelty **at the tip token** (the
  non-Latin onset — where the model switches language).

Then it compares the **tipped** class (language-switching) against a **control** class
(a fluent behaviour), and asks: does the language switch coincide with a routing shift —
a different set of experts lighting up?

**Only local can do this** — OpenRouter returns no router internals. Needs the model on a
single **A100-80GB** (FP8 dequantizes to bf16 ~70 GB). No draft, no DeepSpec: it reuses the
`weird_transcripts.jsonl` + `weird_meta.jsonl` you already have on Drive. One forward per
transcript, resumable.


In [ ]:
# === Cell 1 — config =========================================================
import os
try:
    from google.colab import userdata
    for k in ("HF_TOKEN",):
        v=None
        try: v=userdata.get(k)
        except Exception: v=None
        if v: os.environ.setdefault(k,v)
except Exception as e:
    print("colab secrets unavailable:", e)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")

MODEL   = os.environ.get("WEIRDSPEC_TARGET_MODEL","Qwen/Qwen3.6-35B-A3B-FP8")
DATA_DIR = "/content/drive/MyDrive/weirdspec"     # holds weird_transcripts.jsonl + weird_meta.jsonl (incl. subfolders)
TIPPED_BEHAVIORS  = ["language-switching-english"]      # has a linguistic tip
CONTROL_BEHAVIORS = ["chemtrails-assertion"]            # fluent weird, no switch (the contrast)
N_PER_CLASS = 60          # transcripts per class (one forward each)
MAX_LEN     = 2048
FOREIGN_RUN = 3
NOVELTY_WINDOW = 16
OUTPUT  = "/content/drive/MyDrive/weirdspec/expert_fingerprint.jsonl"   # resumable
MOUNT_DRIVE = True
print("model:", MODEL)


In [ ]:
# === Cell 2 — mount + deps (transformers pin for the qwen3.6 MoE) ===========
import os, sys, subprocess
if MOUNT_DRIVE:
    try:
        from google.colab import drive; drive.mount("/content/drive")
    except Exception as e: print("drive mount skipped:", e)
# The FP8 qwen3.6 MoE loads under the same transformers the pipeline validated.
subprocess.run([sys.executable,"-m","pip","install","-q","transformers==5.10.2","accelerate"], check=True)
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    p=torch.cuda.get_device_properties(0); print(f"GPU: {p.name} {p.total_memory/1024**3:.0f} GB")


In [ ]:
# === Cell 3 — load the target model + read the MoE config ===================
import torch
from transformers import AutoConfig, AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
cfg = AutoConfig.from_pretrained(MODEL)
tc = getattr(cfg,"text_config",cfg)          # the MoE wraps the real config in text_config
def cfgget(o,*names,default=None):
    for n in names:
        v=getattr(o,n,None)
        if v is not None: return v
    return default
NUM_EXPERTS=cfgget(tc,"num_experts","n_routed_experts","num_local_experts")
TOP_K      =cfgget(tc,"num_experts_per_tok","num_experts_per_token","top_k")
HIDDEN     =cfgget(tc,"hidden_size")
N_LAYERS   =cfgget(tc,"num_hidden_layers")
print(f"MoE: num_experts={NUM_EXPERTS}  top_k={TOP_K}  hidden={HIDDEN}  layers={N_LAYERS}")
assert NUM_EXPERTS and TOP_K, "could not read num_experts / num_experts_per_tok from config"

kwargs={"attn_implementation":"sdpa","device_map":"auto"}
kwargs["dtype"]="auto" if getattr(cfg,"quantization_config",None) is not None else torch.bfloat16
try:
    model=AutoModel.from_pretrained(MODEL, **kwargs)
except ValueError:
    import transformers as tf
    model=getattr(tf,str(cfg.architectures[0])).from_pretrained(MODEL, **kwargs)
model.eval()
DEV=next(model.parameters()).device
print("loaded on", DEV)


In [ ]:
# === Cell 4 — find the gates + hook them ====================================
import re, numpy as np, torch
# The FP8 checkpoint wraps the router in a QUANTIZED linear class, so we match on
# the weight SHAPE [num_experts, hidden] (class-agnostic), not isinstance Linear.
# Exclude attention projections (a GQA k/v proj can coincidentally be num_experts-wide).
def _wshape(mod):
    w=getattr(mod,"weight",None)
    return tuple(w.shape) if (w is not None and hasattr(w,"shape") and w.dim()==2) else None
def is_gate(name, shp, num_experts, hidden):
    if shp not in [(num_experts,hidden),(hidden,num_experts)]: return False
    low=name.lower()
    if any(b in low for b in ("shared","attn","proj")): return False
    return ("gate" in low or "router" in low or "moe" in low)
gates=[]
for name,mod in model.named_modules():
    if is_gate(name, _wshape(mod), NUM_EXPERTS, HIDDEN):
        m=re.search(r"layers\.(\d+)\.", name)
        gates.append((int(m.group(1)) if m else -1, name, mod))
gates.sort(key=lambda g:g[0])
if not gates:
    print("no router matched — dumping MoE-ish modules so we can fix the matcher:")
    for name,mod in model.named_modules():
        low=name.lower()
        if any(k in low for k in ("moe","expert","gate","router","sparse")):
            print(f"  {name}  ->  {type(mod).__name__}  weight={_wshape(mod)}")
    raise AssertionError("no MoE gates found — adjust is_gate() from the dump above")
print(f"found {len(gates)} gates (expected <= {N_LAYERS}); "
      f"class={sorted(set(type(g[2]).__name__ for g in gates))}  example={gates[0][1]}")

_cap={}
def _mkhook(pos):
    def hook(module, inp, out):
        logits = out[0] if isinstance(out,(tuple,list)) else out
        _cap[pos]=torch.topk(logits, TOP_K, dim=-1).indices.detach().to("cpu")
    return hook
for h in [g[2].register_forward_hook(_mkhook(i)) for i,g in enumerate(gates)]:
    pass  # handles kept alive by closure; we never remove them this session

@torch.no_grad()
def route(input_ids):
    """Return R: int array [n_gates, seq, top_k] of selected experts per token."""
    _cap.clear()
    ids=torch.tensor([input_ids], device=DEV)
    model(input_ids=ids, use_cache=False)
    seq=ids.shape[1]
    R=np.empty((len(gates),seq,TOP_K), dtype=np.int32)
    for i in range(len(gates)):
        R[i]=_cap[i].numpy().reshape(-1,TOP_K)[:seq]
    return R
print("router capture ready")


In [ ]:
# === Cell 5 — prompt building, tip location, and the metrics ================
import os, glob, json, re, numpy as np
def read_jsonl(path):
    rows=[]
    with open(path,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
def find_file(d,*names):
    for nm in names:
        p=os.path.join(d,nm)
        if os.path.isfile(p): return p
    for nm in names:
        h=sorted(glob.glob(os.path.join(d,"**",nm),recursive=True),key=len)
        if h: return h[0]
    return None

FOREIGN=[(0x0370,0x03FF),(0x0400,0x04FF),(0x0500,0x052F),(0x0530,0x058F),(0x0590,0x05FF),
         (0x0600,0x06FF),(0x0700,0x074F),(0x0900,0x097F),(0x0E00,0x0E7F),
         (0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
def is_foreign(ch):
    if not ch.isalpha(): return False
    o=ord(ch)
    return False if o<0x0250 else any(a<=o<=b for a,b in FOREIGN)
def foreign_onset_char(text, run=FOREIGN_RUN):
    cnt=0; start=None
    for i,ch in enumerate(text):
        if is_foreign(ch):
            if cnt==0: start=i
            cnt+=1
            if cnt>=run: return start
        elif ch.isalpha():
            cnt=0; start=None
    return None

def build_and_locate(conversations, tokenizer, max_len):
    """Return (input_ids, assistant_positions, tip_pos or None)."""
    full="".join(f"<|im_start|>{t['role']}\n{t['content']}<|im_end|>\n" for t in conversations)
    hdr="<|im_start|>assistant\n"; apos=full.rfind(hdr)
    a_char=apos+len(hdr) if apos>=0 else len(full)
    enc=tokenizer(full, return_offsets_mapping=True, truncation=True, max_length=max_len)
    ids=enc["input_ids"]; offs=enc["offset_mapping"]
    A=[i for i,(s,e) in enumerate(offs) if s>=a_char and e>s]
    tip=None
    on=foreign_onset_char(full[a_char:])
    if on is not None:
        tc=a_char+on
        for i,(s,e) in enumerate(offs):
            if s<=tc<e: tip=i; break
    return ids, A, tip

# ---- metrics on R [n_gates, seq, top_k] ----
def expert_set(R, t):
    return set(int(x) for x in R[:,t,:].reshape(-1))
def breadth(R, A, num_experts):
    if not A: return 0.0
    RA=R[:,A,:]
    return float(np.mean([len(np.unique(RA[g]))/num_experts for g in range(R.shape[0])]))
def novelty_series(R, A, W):
    nov=[]; seen=[]
    for t in A:
        S=expert_set(R,t)
        past=set().union(*seen[-W:]) if seen else set()
        nov.append(len(S-past)/max(len(S),1)); seen.append(S)
    return np.array(nov)
print("helpers ready")


In [ ]:
# === Cell 6 — self-test the metrics on synthetic routing (no model) =========
import numpy as np
G,seq,K,NE=8,60,4,64
rng=np.random.default_rng(0)
R=rng.integers(0,11,size=(G,seq,K))          # tokens use experts {0..10}
R[:,40:,:]=rng.integers(50,61,size=(G,20,K)) # a routing SHIFT at token 40 (experts {50..60})
A=list(range(seq)); nov=novelty_series(R,A,16)
print(f"breadth={breadth(R,A,NE):.3f}  novelty@tip(40)={nov[40]:.2f}  median={np.median(nov):.2f}")
assert nov[40] > 0.8 and np.median(nov[:35]) < 0.3, "novelty should spike at the injected shift"
H=2048
assert is_gate("model.layers.12.mlp.gate", (NE,H), NE, H)                     # the router
assert not is_gate("model.layers.3.mlp.shared_expert_gate", (NE,H), NE, H)    # shared gate excluded
assert not is_gate("model.layers.5.self_attn.k_proj", (NE,H), NE, H)          # GQA k_proj (same shape!) excluded
assert not is_gate("lm_head", (151936,H), NE, H)                              # wrong shape
print("self-test OK: novelty spikes at the routing shift; gate detector is correct")


In [ ]:
# === Cell 7 — teacher-force each transcript, capture routing (resumable) =====
import os, json, numpy as np
wp=find_file(DATA_DIR,"weird_transcripts.jsonl")
mp=find_file(DATA_DIR,"weird_meta.jsonl")
assert wp and mp, f"need weird_transcripts.jsonl + weird_meta.jsonl under {DATA_DIR}"
trans=read_jsonl(wp); meta=read_jsonl(mp)
by_beh={}
for tr,m in zip(trans,meta):
    by_beh.setdefault(m.get("behavior_id"),[]).append((tr,m))
def pick(behs):
    out=[]
    for b in behs: out+=by_beh.get(b,[])[:N_PER_CLASS]
    return out
selection=[("tipped",tr,m) for tr,m in pick(TIPPED_BEHAVIORS)] + \
          [("control",tr,m) for tr,m in pick(CONTROL_BEHAVIORS)]
print(f"tipped={sum(c=='tipped' for c,_,_ in selection)}  control={sum(c=='control' for c,_,_ in selection)}")

done={r["id"] for r in (read_jsonl(OUTPUT) if os.path.exists(OUTPUT) else [])}
outf=open(OUTPUT,"a",encoding="utf-8")
for n,(cls,tr,m) in enumerate(selection,1):
    if tr["id"] in done: continue
    try:
        ids,A,tip=build_and_locate(tr["conversations"], tokenizer, MAX_LEN)
        if len(A)<4: continue
        R=route(ids)
        nov=novelty_series(R,A,NOVELTY_WINDOW)
        tip_in_A=A.index(tip) if (tip is not None and tip in A) else None
        rec=dict(id=tr["id"], cls=cls, behavior=m.get("behavior_id"),
                 seq=len(ids), n_assistant=len(A), has_tip=tip_in_A is not None,
                 breadth=breadth(R,A,NUM_EXPERTS),
                 median_novelty=float(np.median(nov)), max_novelty=float(nov.max()),
                 tip_novelty=(float(nov[tip_in_A]) if tip_in_A is not None else None),
                 tip_frac=(tip_in_A/len(A) if tip_in_A is not None else None),
                 novelty=[round(float(x),3) for x in nov[:400]])
        outf.write(json.dumps(rec,ensure_ascii=False)+"\n"); outf.flush()
    except Exception as e:
        print(f"  [{n}] {tr['id'][:24]} error: {type(e).__name__}: {e}")
    if n%20==0: print(f"  {n}/{len(selection)}")
outf.close(); print("saved ->", OUTPUT)


In [ ]:
# === Cell 8 — compare tipped vs control + tip routing-shift =================
# Self-contained: runs on the saved expert_fingerprint.jsonl alone — no model
# needed. In a fresh session just run Cell 1, then this cell.
import os, json
import numpy as np, matplotlib.pyplot as plt
if not os.path.exists(OUTPUT):
    try:
        from google.colab import drive; drive.mount("/content/drive")
    except Exception as e: print("drive mount skipped:", e)
def _read(path):
    rows=[]
    with open(path,encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    return rows
rows=_read(OUTPUT)
tip=[r for r in rows if r["cls"]=="tipped"]; ctl=[r for r in rows if r["cls"]=="control"]
print(f"tipped={len(tip)}  control={len(ctl)}")
def boot_ci(a,b,f,n=3000,seed=0):
    rng=np.random.default_rng(seed); a=np.array(a); b=np.array(b)
    d=[f(rng.choice(a,len(a)))-f(rng.choice(b,len(b))) for _ in range(n)]
    return f(a)-f(b), np.percentile(d,2.5), np.percentile(d,97.5)

# (1) expert breadth: does the tipped mode use a wider/narrower expert set?
bt=[r["breadth"] for r in tip]; bc=[r["breadth"] for r in ctl]
if bt and bc:
    d,lo,hi=boot_ci(bt,bc,np.mean)
    print(f"\nexpert breadth  tipped {np.mean(bt):.3f}  control {np.mean(bc):.3f}  "
          f"diff {d:+.3f} [{lo:+.3f},{hi:+.3f}]")

# (2) the key test: at the tip token, is routing novelty ABOVE the transcript's baseline?
paired=[(r["tip_novelty"]-r["median_novelty"]) for r in tip if r.get("has_tip")]
if len(paired)>=8:
    pa=np.array(paired); rng=np.random.default_rng(1)
    bs=[np.mean(rng.choice(pa,len(pa))) for _ in range(3000)]
    lo,hi=np.percentile(bs,[2.5,97.5])
    print(f"\ntip routing shift: novelty(tip) - median = {pa.mean():+.3f}  95% CI [{lo:+.3f},{hi:+.3f}]  (n={len(pa)})")
    print("   >0 with CI excluding 0  =>  the language switch coincides with a routing shift (new experts fire)")
else:
    print(f"\nonly {len(paired)} tipped transcripts have a located tip — too few for the shift test.")

# ---- event-aligned novelty around the tip -----------------------------------
# The first NOVELTY_WINDOW tokens are trivially "novel" (empty history); that
# burn-in transient is what made the raw traces unreadable. Here we drop it and
# align every tipped transcript to its switch token (x = token - tip), then show
# the mean with a 95% band — against a control baseline aligned to
# distribution-matched pseudo-tips.
W=NOVELTY_WINDOW; SPAN=60
def tip_index(r):
    if not r.get("has_tip") or r.get("tip_frac") is None: return None
    ti=int(round(r["tip_frac"]*r["n_assistant"]))
    return ti if W<=ti<len(r["novelty"]) else None
rngp=np.random.default_rng(2)
tip_fracs=[r["tip_frac"] for r in tip if r.get("has_tip") and r.get("tip_frac") is not None]
def pseudo_index(r):
    if not tip_fracs: return None
    ti=int(round(rngp.choice(tip_fracs)*r["n_assistant"]))
    return ti if W<=ti<len(r["novelty"]) else None
def aligned(rs, indexer):
    M=np.full((len(rs),2*SPAN+1),np.nan)
    for i,r in enumerate(rs):
        ti=indexer(r)
        if ti is None: continue
        nov=r["novelty"]
        for dt in range(-SPAN,SPAN+1):
            t=ti+dt
            if W<=t<len(nov): M[i,dt+SPAN]=nov[t]
    return M[~np.isnan(M).all(axis=1)]
def mean_band(M):
    n=np.maximum((~np.isnan(M)).sum(0),1)
    mu=np.nanmean(M,0); se=np.nanstd(M,0)/np.sqrt(n)
    return mu, mu-1.96*se, mu+1.96*se
MT=aligned(tip,tip_index); MC=aligned(ctl,pseudo_index)
x=np.arange(-SPAN,SPAN+1)
print(f"aligned: {MT.shape[0]} tipped, {MC.shape[0]} control (burn-in <{W} removed)")

fig,ax=plt.subplots(1,2,figsize=(11,4))
# left: breadth distributions with jittered points (identity via position, not color)
if bt and bc:
    ax[0].boxplot([bt,bc],showfliers=False)   # labels kw renamed across mpl versions
    ax[0].set_xticks([1,2]); ax[0].set_xticklabels(["tipped","control"])
    for k,vals in enumerate([bt,bc],1):
        ax[0].plot(k+(np.random.default_rng(4).uniform(-.08,.08,len(vals))),vals,
                   "o",ms=3,alpha=.4,color="#2563EB" if k==1 else "#6B7280")
    ax[0].set_ylabel("distinct-expert breadth"); ax[0].set_title("expert breadth per transcript")
# right: event-aligned mean ± 95% band, faint individual tipped traces behind
if MT.shape[0]:
    for row in MT[:40]:
        ax[1].plot(x,row,color="#9CA3AF",alpha=.12,lw=.6)
    mu,lo_,hi_=mean_band(MT)
    ax[1].fill_between(x,lo_,hi_,color="#2563EB",alpha=.20,linewidth=0)
    ax[1].plot(x,mu,color="#2563EB",lw=2,label="tipped (aligned to switch)")
if MC.shape[0]:
    mu2,lo2,hi2=mean_band(MC)
    ax[1].fill_between(x,lo2,hi2,color="#6B7280",alpha=.15,linewidth=0)
    ax[1].plot(x,mu2,color="#6B7280",lw=2,ls="--",label="control (pseudo-tip)")
ax[1].axvline(0,color="black",lw=.8,ls=":")
ax[1].set_xlabel("token relative to the language switch"); ax[1].set_ylabel("routing novelty")
ax[1].set_title("routing novelty around the tip (mean ± 95% CI)")
ax[1].set_ylim(bottom=0); ax[1].legend(frameon=False,fontsize=9)
for a in ax: a.grid(True,alpha=.25,linewidth=.5)
plt.tight_layout(); plt.show()


### How to read it

* **Expert breadth** — if the tipped (language-switching) transcripts use a
  significantly different number of distinct experts than the fluent control, the
  weird mode has its own routing signature (wider = more scattered, narrower = a
  few dominant experts carrying the switch).
* **Tip routing shift** — the headline. `novelty(tip) − median` positive with a CI
  excluding 0 means that at the exact token where the language flips, a *new* set
  of experts lights up that wasn't being used just before — a routing discontinuity
  coincident with the behavioural tip. That is the MoE-level correlate of the
  "tipped state" the HMM found and the draft-surprise peak localised.
* **The event-aligned plot** is the visual of the same thing: every tipped
  transcript aligned to its switch token (x = 0), burn-in removed, mean ± 95% band —
  a bump at 0 that the pseudo-tip-aligned control baseline lacks. Faint gray lines
  are the individual tipped transcripts.

Needs the A100-80GB (FP8 → bf16 ~70 GB). One forward per transcript; resumable via
the `id` set, so a Colab disconnect just means re-running Cell 7.
